# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZubairQazzi/flyrank-ml-internship-zubair/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

### Feature vector

My lane is **content refresh opportunity scoring**.

The unit of analysis is one published content page. I build five pre-decision search features from March 1–21, 2026 and reserve March 22–31, 2026 only for defining the later `future_decline` proxy.

The five features are:

- `past_impressions`
- `past_clicks`
- `past_ctr`
- `past_avg_position`
- `gsc_observed_days`

Client and content IDs are retained only as pseudonymous context fields for grouping and joins. They are not model features.

In [1]:
from google.colab import userdata
import duckdb
import pandas as pd
import numpy as np

hf_token = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"""
    CREATE SECRET hf_secret (
        TYPE huggingface,
        TOKEN '{hf_token}'
    )
    """
)

march_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/*.parquet"
)

content_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "dim_content.parquet"
)

feature_query = f"""
WITH daily_windows AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-21'
                 AND gsc_data_available IS TRUE
                THEN gsc_impressions
                ELSE 0
            END
        ) AS past_impressions,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-21'
                 AND gsc_data_available IS TRUE
                THEN gsc_clicks
                ELSE 0
            END
        ) AS past_clicks,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-21'
                 AND gsc_data_available IS TRUE
                THEN gsc_sum_position
                ELSE 0
            END
        ) AS past_sum_position,

        COUNT(*) FILTER (
            WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-21'
              AND gsc_data_available IS TRUE
        ) AS gsc_observed_days,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-22' AND DATE '2026-03-31'
                 AND gsc_data_available IS TRUE
                THEN gsc_impressions
                ELSE 0
            END
        ) AS outcome_impressions,

        COUNT(*) FILTER (
            WHERE report_date BETWEEN DATE '2026-03-22' AND DATE '2026-03-31'
              AND gsc_data_available IS TRUE
        ) AS outcome_days

    FROM read_parquet('{march_path}')
    GROUP BY client_hash_id, content_hash_id
),

content_status AS (
    SELECT
        client_hash_id,
        content_hash_id,
        is_published,
        is_deleted
    FROM read_parquet('{content_path}')
)

SELECT
    d.client_hash_id,
    d.content_hash_id,
    d.past_impressions,
    d.past_clicks,

    ROUND(
        100.0 * d.past_clicks /
        NULLIF(d.past_impressions, 0),
        4
    ) AS past_ctr,

    ROUND(
        1.0 * d.past_sum_position /
        NULLIF(d.past_impressions, 0),
        4
    ) AS past_avg_position,

    d.gsc_observed_days,

    CASE
        WHEN
            (1.0 * d.outcome_impressions / d.outcome_days)
            <
            0.80 * (
                1.0 * d.past_impressions /
                d.gsc_observed_days
            )
        THEN 1
        ELSE 0
    END AS future_decline

FROM daily_windows d

INNER JOIN content_status c
    ON d.client_hash_id = c.client_hash_id
   AND d.content_hash_id = c.content_hash_id

WHERE
    c.is_published IS TRUE
    AND COALESCE(c.is_deleted, FALSE) IS FALSE
    AND d.gsc_observed_days >= 7
    AND d.outcome_days >= 5
    AND d.past_impressions >= 100
"""

feature_frame = con.sql(feature_query).df()

feature_frame = (
    feature_frame
    .sort_values(["client_hash_id", "content_hash_id"])
    .reset_index(drop=True)
)

features = [
    "past_impressions",
    "past_clicks",
    "past_ctr",
    "past_avg_position",
    "gsc_observed_days",
]

print("Rows in feature frame:", len(feature_frame))
print("Unique clients:", feature_frame["client_hash_id"].nunique())
print("\nLabel distribution:")
print(feature_frame["future_decline"].value_counts())

display(
    feature_frame[
        features + ["future_decline"]
    ].head()
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows in feature frame: 85967
Unique clients: 39

Label distribution:
future_decline
0    58038
1    27929
Name: count, dtype: int64


,past_impressions,past_clicks,past_ctr,past_avg_position,gsc_observed_days,future_decline
0,190.0,1.0,0.5263,12.2316,21,0
1,353.0,0.0,0.0000,13.4051,21,1
2,106.0,0.0,0.0000,12.8019,19,0
3,133.0,0.0,0.0000,9.5714,8,0
4,627.0,3.0,0.4785,12.4593,21,1


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

### Feature notes

All five model features are numeric and are calculated only from information available before the March 21, 2026 decision moment.

- `past_impressions` — total observed GSC impressions from March 1–21. Missing search observations are not treated as evidence; only days where `gsc_data_available IS TRUE` contribute.
- `past_clicks` — total observed GSC clicks from March 1–21 under the same availability rule.
- `past_ctr` — calculated as past clicks divided by past impressions. The population filter requires at least 100 past impressions, so the denominator is non-zero in the final frame.
- `past_avg_position` — calculated from observed GSC position information in the feature window only.
- `gsc_observed_days` — number of feature-window days with usable GSC data. The final frame requires at least 7 observed days.

No categorical feature encoding is required because all five predictive features are numeric.

`client_hash_id` and `content_hash_id` are pseudonymous context fields used for grouping and joins only. They are not predictive inputs.

The later March 22–31 window is used only to construct the evaluation proxy and is not available to the model at prediction time.

In [2]:
# Verify feature types, missing values, and basic ranges

feature_notes = pd.DataFrame(
    {
        "feature": features,
        "dtype": [
            str(feature_frame[col].dtype)
            for col in features
        ],
        "missing_values": [
            int(feature_frame[col].isna().sum())
            for col in features
        ],
        "minimum": [
            float(feature_frame[col].min())
            for col in features
        ],
        "maximum": [
            float(feature_frame[col].max())
            for col in features
        ],
    }
)

print("Predictive features:", features)
print(
    "IDs used as features:",
    any(
        col in features
        for col in ["client_hash_id", "content_hash_id"]
    ),
)
print(
    "Future label used as feature:",
    "future_decline" in features,
)

display(feature_notes)

Predictive features: ['past_impressions', 'past_clicks', 'past_ctr', 'past_avg_position', 'gsc_observed_days']
IDs used as features: False
Future label used as feature: False


,feature,dtype,missing_values,minimum,maximum
0,past_impressions,float64,0,100.000,273012.0000
1,past_clicks,float64,0,0.000,3398.0000
2,past_ctr,float64,0,0.000,18.7500
3,past_avg_position,float64,0,0.023,115.3731
4,gsc_observed_days,int64,0,7.000,21.0000


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*


### Leakage hunt

I test the feature set from two directions.

First, I verify that the final predictive feature list does not contain identifiers, future-window fields, label-derived trend fields, product decision outputs, or the target itself.

Second, I deliberately create an impossible feature called `label_leak` that directly copies `future_decline`. This represents the type of target leakage that would make validation results look unrealistically strong.

The honest and deliberately leaked models use the same grouped client split. If the leaked model becomes nearly perfect while the honest model remains modest, that demonstrates why target-derived information must never enter the real feature set.

After the experiment, the leaking column is removed.

In [3]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# --------------------------------------------------
# Check the final feature names for obvious leakage
# --------------------------------------------------

forbidden_exact = {
    "client_hash_id",
    "content_hash_id",
    "future_decline",
    "trend_direction",
    "trend_pct",
    "priority_score",
    "action_type",
    "health_score",
}

forbidden_used = [
    col for col in features
    if col in forbidden_exact
]

future_like_used = [
    col for col in features
    if (
        col.startswith("future_")
        or col.startswith("outcome_")
    )
]

print("Forbidden exact fields used:", forbidden_used)
print("Future/outcome fields used:", future_like_used)


# --------------------------------------------------
# Same grouped client split for both experiments
# --------------------------------------------------

X_honest = feature_frame[features].astype(float)
y = feature_frame["future_decline"].astype(int)
groups = feature_frame["client_hash_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42,
)

train_idx, test_idx = next(
    splitter.split(
        X_honest,
        y,
        groups=groups,
    )
)


def calculate_auc(X):
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
        ),
    )

    model.fit(
        X.iloc[train_idx],
        y.iloc[train_idx],
    )

    probabilities = model.predict_proba(
        X.iloc[test_idx]
    )[:, 1]

    return roc_auc_score(
        y.iloc[test_idx],
        probabilities,
    )


# --------------------------------------------------
# Honest model
# --------------------------------------------------

honest_auc = calculate_auc(X_honest)


# --------------------------------------------------
# Deliberately leaked model
# --------------------------------------------------

X_leaked = X_honest.copy()
X_leaked["label_leak"] = y

leaked_auc = calculate_auc(X_leaked)


# --------------------------------------------------
# Remove leakage and verify recovery
# --------------------------------------------------

X_final = X_leaked.drop(
    columns=["label_leak"]
)

honest_auc_after_removal = calculate_auc(
    X_final
)

print("\nHonest ROC-AUC:", round(honest_auc, 4))
print("Leaked ROC-AUC:", round(leaked_auc, 4))
print(
    "Honest ROC-AUC after removing leakage:",
    round(honest_auc_after_removal, 4),
)
print(
    "Leak column still present:",
    "label_leak" in X_final.columns,
)

Forbidden exact fields used: []
Future/outcome fields used: []

Honest ROC-AUC: 0.5537
Leaked ROC-AUC: 1.0
Honest ROC-AUC after removing leakage: 0.5537
Leak column still present: False


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

### Excluded fields

I deliberately exclude fields that would make the model unsafe, non-generalizable, or impossible to use at the real decision moment.

- `client_hash_id` — pseudonymous identifier used only for grouping and joins. It should not become a predictive shortcut.
- `content_hash_id` — pseudonymous content identifier used only for grouping and tracing rows.
- `future_decline` — the target itself, so using it as a feature would be direct leakage.
- `trend_direction` and `trend_pct` — label-derived trend fields that may encode information closely related to the outcome.
- future-window and outcome-window fields — these are not available at the March 21 decision moment.
- `priority_score`, `action_type`, and `health_score` — product decision outputs should not be reused as ordinary predictive features.
- client names, URLs, raw queries, titles, domains, and other identifying text — excluded for privacy and public-safety reasons.
- GA4 fields — excluded from this first feature frame because availability differs by client and unavailable values may be represented as zeros.

The final predictive feature list therefore contains only five numeric, pre-decision search-performance features.

In [5]:
excluded_fields = pd.DataFrame(
    [
        {
            "field": "client_hash_id",
            "reason": "Grouping/join context only; not a predictive feature.",
        },
        {
            "field": "content_hash_id",
            "reason": "Grouping/tracing context only; not a predictive feature.",
        },
        {
            "field": "future_decline",
            "reason": "Target variable; using it would be direct leakage.",
        },
        {
            "field": "trend_direction / trend_pct",
            "reason": "Label-derived trend information may leak outcome information.",
        },
        {
            "field": "future_* / outcome_*",
            "reason": "Not available at the prediction moment.",
        },
        {
            "field": "priority_score / action_type / health_score",
            "reason": "Product decision outputs should not become model shortcuts.",
        },
        {
            "field": "client names / URLs / raw queries",
            "reason": "Excluded for privacy and public safety.",
        },
        {
            "field": "GA4 fields",
            "reason": "Availability differs by client and unavailable values may be zero-filled.",
        },
    ]
)

print("Final predictive feature count:", len(features))
print("Final predictive features:")
print(features)

print(
    "\nAny forbidden fields in final feature list:",
    bool(forbidden_used or future_like_used),
)

display(excluded_fields)

Final predictive feature count: 5
Final predictive features:
['past_impressions', 'past_clicks', 'past_ctr', 'past_avg_position', 'gsc_observed_days']

Any forbidden fields in final feature list: False


,field,reason
0,client_hash_id,Grouping/join context only; not a predictive f...
1,content_hash_id,Grouping/tracing context only; not a predictiv...
2,future_decline,Target variable; using it would be direct leak...
3,trend_direction / trend_pct,Label-derived trend information may leak outco...
4,future_* / outcome_*,Not available at the prediction moment.
5,priority_score / action_type / health_score,Product decision outputs should not become mod...
6,client names / URLs / raw queries,Excluded for privacy and public safety.
7,GA4 fields,Availability differs by client and unavailable...


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.